# Vector Stores and Retrievers

Retriever abstractions are designed to support retrieval of data from (vector) databases and other sources for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augumented generation.

## Documents

LangChain implements a document abstraction, which is intended to represent a unit of text and associated metadata. 

It has two attributes:

- **page_content:** a string representing the content
- **metadata:** a dict containing arbitrary metadata. The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual document object often represents a chunk of a larger document.

In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for thier loyality and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy thier own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
]

In [2]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for thier loyality and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key, model="Llama3-8b-8192")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x110e89280>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1110d72c0>, model_name='Llama3-8b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
## Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/Users/rahulsaini/Documents/repositories/gen-ai-cookbook/04-gen-ai/langchain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
## VectorStores
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding=embeddings)
vectorstore

In [8]:
## async query
await vectorstore.asimilarity_search("cat")

[Document(id='1c7313e1-12ce-4130-85b4-8a76cffdf821', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space.'),
 Document(id='7e817646-7aa1-43d9-9ce5-1cfdfa418ef5', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for thier loyality and friendliness.'),
 Document(id='7c83cecd-3b37-49eb-8c06-3ae22461f108', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(id='3dafa3b3-52df-421e-8cad-cb1b6dcfb0a4', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')]

## Retrivers

LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language(LCEL) chains.

LangChain Retrieveres are Runnables, so they implement a standard set of methods (e.g, synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

In [9]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat", "dog"])

[[Document(id='1c7313e1-12ce-4130-85b4-8a76cffdf821', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space.')],
 [Document(id='7e817646-7aa1-43d9-9ce5-1cfdfa418ef5', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for thier loyality and friendliness.')]]

VectorStores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them.

Implementing the above using vectorstore.

In [10]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})
retriever.batch(["cat", "dog"])

[[Document(id='1c7313e1-12ce-4130-85b4-8a76cffdf821', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy thier own space.')],
 [Document(id='7e817646-7aa1-43d9-9ce5-1cfdfa418ef5', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for thier loyality and friendliness.')]]

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.
{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

response = rag_chain.invoke("tell me about dogs")
print(response.content)

According to the provided context, dogs are great companions, known for their loyalty and friendliness.
